# IRP: Evaluating how Crime is affected by access to Public Transport vs Green Space in Sydney, Australia

* **Authors:** Beatriz Pinol

* **Student ID:** 220002252

* **Date:** 30/04/25

**Abstract:**

[Write a concise summary of your research here.]

**Keywords:** Green Space, Urban, Crime, Public Transport, 

# GitHub Repository
- **GitHub Link:** https://github.com/beapinol/UrbanAn

## Declaration

> In submitting this assignment, I hereby confirm that I have read the University's statement on Good Academic Practice. The following work is my own. Significant academic debts and borrowings have been properly acknowledged and referenced.


**Table of Contents:** 

* Section 1
* Section 2
* Section n


## Introduction

Provide a clear introduction to your research, outlining the problem you're addressing,  the research question, its significance, and relevant background information. Include citations using appropriate formatting (e.g., BibTeX, Markdown footnotes).


## Methodology

Describe the methods used in your research, including data sources, collection procedures, analysis techniques, and any relevant software tools. Include code cells for data loading, preprocessing, and key steps in your analysis pipeline.


In [ ]:
#Importing the necessary libraries
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from shapely.ops import unary_union

In [ ]:
#loading in my villages data 
villages_path = 'data/villages/Our_villages.shp'
villages = gpd.read_file(villages_path)

In [ ]:
villages.head()

In [ ]:
villages.plot()

In [ ]:
#loading in my parks data
parks_path = 'data/parks/Parks.shp'
parks = gpd.read_file(parks_path)

In [ ]:
parks.head()

In [ ]:
parks.plot()

In [ ]:
#loading in my trees data
trees_path = 'data/trees/Trees.shp'
trees = gpd.read_file(trees_path)

In [ ]:
trees.head()

In [ ]:
trees.plot()

In [ ]:
trees.explore()

In [ ]:
#loading in bus shelters data
bus_shelters_path = 'data/bus_shelters/Bus_shelters.shp'
bus_shelters = gpd.read_file(bus_shelters_path)

In [ ]:
bus_shelters.head()

In [ ]:
bus_shelters.plot()

In [ ]:
bus_shelters.explore()

In [ ]:
#loading in station entrances data 
station_entrances = pd.read_csv('data/station_entrances/station_entrances.csv')

In [ ]:
station_entrances.head()

In [ ]:
#loading crime data 
outdoor_crime_path = 'data/outdoor_crime/outdoor_crime.csv'
outdoor_crime = gpd.read_file(outdoor_crime_path)

In [ ]:
outdoor_crime.head()

In [ ]:
outdoor_crime.columns

In [ ]:
#here i need to rename the columns

In [ ]:
rename_cols = {
    'field_1': 'Feature_ID',
    'field_2': 'OBJECTID',
    'field_3': 'CrimeGrpCode',
    'field_4': 'CrimeCat',
    'field_5': 'LGAname',
    'field_6': 'Suburb',
    'field_7': 'PrimaryLocation',
    'field_8': 'PostalCode',
    'field_9': 'LAT',
    'field_10': 'LONG',
    'field_11': 'GeocodeSource',
    'field_12': 'Year',
    'field_13': 'Month',
    'field_14': 'Day',
    'field_15': 'StartTime',
    'field_16': 'EventYear',
    'field_17': 'POI_Sex',
    'field_18': 'POI_Age',
    'field_19': 'UniqueID'
}
outdoor_crime.rename(columns=rename_cols, inplace=True)

In [ ]:
#checking that our renaming of the columns was successful
outdoor_crime.head()

In [ ]:
#preparing the data for analysis

In [ ]:
#standardising the crs of data to web mercator (EPSG:3857)
villages = villages.to_crs(epsg=3857)
parks = parks.to_crs(epsg=3857)
trees = trees.to_crs(epsg=3857)
bus_shelters = bus_shelters.to_crs(epsg=3857)

In [ ]:
#converting stations to GeoDataFrame and reproject
stations_gdf = gpd.GeoDataFrame(
    station_entrances, 
    geometry=gpd.points_from_xy(station_entrances['LONG'], station_entrances['LAT']),
    crs='EPSG:4326'
).to_crs(epsg=3857)

In [ ]:
#converting the crime csv to GeoDataFrame and reprojecting it to the same CRS
outdoor_crime['LAT'] = pd.to_numeric(outdoor_crime['LAT'], errors='coerce')
outdoor_crime['LONG'] = pd.to_numeric(outdoor_crime['LONG'], errors='coerce')
outdoor_crime = outdoor_crime.dropna(subset=['LONG', 'LAT'])
crime_gdf = gpd.GeoDataFrame(
    outdoor_crime,
    geometry=gpd.points_from_xy(outdoor_crime['LONG'], outdoor_crime['LAT']),
    crs='EPSG:4326'
).to_crs(epsg=3857)

In [ ]:
#calculating green space metrics: park counts per village

In [ ]:
village_park_counts = [parks.intersects(poly).sum() for poly in villages.geometry]
villages['park_count'] = village_park_counts

In [ ]:
#calculating tree counts per village

In [ ]:
villages['tree_count'] = villages.geometry.apply(lambda poly: trees.intersects(poly).sum())

In [ ]:
#calculating transport accessiblity: flag within 2500m of a bus stop

In [ ]:
all_stops = pd.concat([bus_shelters, stations_gdf], ignore_index=True)
buffer_union = unary_union(all_stops.buffer(2500))
villages['transport_access'] = villages.geometry.intersects(buffer_union).astype(int)

In [ ]:
#crime analysis - per village area

In [ ]:
joined_crime = gpd.sjoin(crime_gdf, villages, how='left', predicate='within')
crime_counts = joined_crime.groupby('Name').size().reset_index(name='crime_count')
villages = villages.merge(crime_counts, on='Name', how='left').fillna({'crime_count': 0})

In [ ]:
#computing minimum distance from each crime to nearest park

In [ ]:
park_union = unary_union(parks.geometry)
crime_gdf['dist_to_park'] = crime_gdf.geometry.apply(lambda pt: pt.distance(park_union))

In [ ]:
#cleaning out infinite or invalid distances from crime_gdf 
crime_gdf['dist_to_park'] = crime_gdf['dist_to_park'].replace([np.inf, -np.inf], np.nan)
crime_gdf = crime_gdf.dropna(subset=['dist_to_park'])

In [ ]:
#flagging crimes within buffer distance of any park

## Results

Illustrate your findings organised, using clear and concise text, informative tables, and good visualizations. Explain the meaning and significance of your results, making connections to the research questions or any hypotheses outlined in the introduction.


In [1]:
#chloropleth map plot of crime and park count per village

In [ ]:
#park count chloropleth map
fig, ax = plt.subplots(figsize=(8,6))
villages.plot(column='park_count', cmap='Greens', legend=True, ax=ax)
ax.set_title('Park Count per Village')
ax.axis('off')
plt.show()

In [ ]:
#tree count chloropleth map
fig, ax = plt.subplots(figsize=(8,6))
villages.plot(column='tree_count', cmap='Greens', legend=True, ax=ax)
ax.set_title('Tree Count per Village')
ax.axis('off')
plt.show()

## Discussion

Interpret your results in the context of existing literature. Discuss any limitations of your study and potential future research directions.


## Conclusion

Summarize the key findings and their implications. Briefly restate the research question and highlight the main contributions of your project.


## Appendix

Include any additional information and code not included in the main body of the paper but may be necessary. This could include:

* Raw data tables
* Additional figures or tables
* Supporting code


## References

You could use an appropriate citation management tool (e.g., Zotero, Mendeley) to create BibTeX entries for your references. Paste the generated BibTeX code here, and you can use tools like `pandoc` (if necessary) to convert it to the desired reference format (e.g., APA, MLA).
